# ByteCover ablation

First pass only. These cells put the cached CQTs on this machine and check them against the manifest. They do not train a model.

The zip stays in Google Drive at `ByteCover Project/cqt.zip`. On Colab, the notebook unzips it to `/content/cqt`. That disk is wiped when the runtime stops, so a later session unzips the same zip again. The unpacked files are not uploaded back to Drive.

A CPU runtime is enough for this check. Use a GPU only when training starts.

The archive already contains a top-level `cqt/` folder, so extracting it into `/content` produces `/content/cqt/shs/` and `/content/cqt/covers80/`.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/wishang06/bytecover-research-project.git"

if Path("data/train_manifest.csv").exists():
    ROOT = Path(".").resolve()
else:
    ROOT = Path("/content/bytecover-research-project")
    if not (ROOT / "data/train_manifest.csv").exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(ROOT)])
print("repo:", ROOT)


In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/ByteCover Project/cqt.zip")

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    if not ZIP_PATH.exists():
        print(f"zip not found: {ZIP_PATH}")
        parent = ZIP_PATH.parent.parent
        if parent.exists():
            print("folders in MyDrive:")
            for folder in sorted(parent.iterdir()):
                print(" ", folder.name)
        raise SystemExit(1)
    cqt_root = Path("/content/cqt")
    if not (cqt_root / "shs").exists():
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall("/content")
    print("colab cache:", cqt_root)
else:
    cqt_root = ROOT / "cqt"
    print("local cache:", cqt_root)


In [ ]:
import csv

manifest_path = ROOT / "data" / "train_manifest.csv"
with manifest_path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

kept = [row for row in rows if row["excluded"] == "0"]
works = {row["work_id"] for row in kept}
missing = [
    row["perf_id"]
    for row in kept
    if not (cqt_root / "shs" / f"{row['perf_id']}.npy").is_file()
]
covers = list((cqt_root / "covers80").glob("*.npy"))

print(f"manifest rows: {len(rows)}")
print(f"kept recordings: {len(kept)}")
print(f"kept works: {len(works)}")
print(f"missing shs npy: {len(missing)}")
print(f"covers80 npy: {len(covers)}")
if missing[:5]:
    print("missing examples:", missing[:5])

ok = (
    not missing
    and len(rows) == 1498
    and len(kept) == 1498
    and len(works) == 300
    and len(covers) == 164
)
print("PASS" if ok else "FAIL")
if not ok:
    raise SystemExit(1)
